## Tylko testy do większych zbiorów danych

In [1]:
import os
import pandas as pd
from torchvision.io import read_image
import re
import wfdb
import wfdb.processing
import scipy
from torch.utils.data import Dataset
import numpy as np
import json
import torch.nn as nn
import torch
from tqdm import tqdm
import torch.nn.functional as F

In [2]:
def extract_segment_with_padding(z, k, N):
    # Rozmiar segmentu to 2N + 1
    start_idx = k - N
    end_idx = k + N + 1  # Indeks końcowy +1, ponieważ Python używa wykluczającego indeksu
    
    # Upewnij się, że start_idx i end_idx mieszczą się w granicach tablicy
    if start_idx < 0:
        # Jeśli start_idx jest poza zakresem, dopełnij na początku
        padding_left = np.median(z[:end_idx])  # Wypełniamy medianą
        segment = np.concatenate([np.full(-start_idx, padding_left), z[:end_idx]])
    elif end_idx > len(z):
        # Jeśli end_idx jest poza zakresem, dopełnij na końcu
        padding_right = np.median(z[start_idx:])  # Wypełniamy medianą
        segment = np.concatenate([z[start_idx:], np.full(end_idx - len(z), padding_right)])
    else:
        # Normalny przypadek, kiedy zakres mieści się w tablicy
        segment = z[start_idx:end_idx]
    
    return segment

def find_nearest_qrs_index(annotation_sample, qrs_inds):
    # Find the index in qrs_inds that is closest to annotation_sample
    distances = np.abs(qrs_inds - annotation_sample)
    nearest_idx = np.argmin(distances)  # Get the index of the minimum distance
    return qrs_inds[nearest_idx]

class SHDB(Dataset):
    def __init__(self,N, M, dataset_dir = 'Datasets/temp2/shdb-af-a-japanese-holter-ecg-database-of-atrial-fibrillation-1.0.0/shdb-af-a-japanese-holter-ecg-database-of-atrial-fibrillation-1.0.0/', fs = 10):
        """
        n - number of samples of orginal signal resampled to fs, interval [-n,n]
        m - qrs times, interval [-m,m]
        """
        self.N = N
        self.ecg_list = []
        exclusion_lst = []
        for file in os.listdir(dataset_dir):
            name = re.match(r'^(.*\d\d+)\.atr$', file)
            if name:
                if name.group(1) in exclusion_lst:
                    continue
            if name:
                print(name.group(1))
                record = wfdb.rdsamp(f"{dataset_dir}{name.group(1)}") 
                annotation = wfdb.rdann(f"{dataset_dir}{name.group(1)}", 'atr')
                signal = record[0][:,0]
                fs_original = record[1]["fs"]
                num_samples_target = int(signal.shape[0] * fs / fs_original)
                resampled_signal = scipy.signal.resample(signal, num_samples_target)
                annotation_times_resampled = (annotation.sample * fs) / fs_original
                resampled_annotation = wfdb.Annotation('atr',annotation.symbol,annotation_times_resampled.astype(int),aux_note=annotation.aux_note)
                
                labels = resampled_annotation.aux_note
                segments = resampled_annotation.sample

                filtered_labels = []
                filtered_segments = []  # Start with the first segment boundary

                for i, label in enumerate(labels):
                    if label.strip():  # Include only non-empty labels
                        filtered_labels.append(label)
                        filtered_segments.append(segments[i])
                resampled_annotation.aux_note = np.array(filtered_labels)
                resampled_annotation.sample = np.array(filtered_segments)
                self.ecg_list.append({"name": name.group(1),"rec" : resampled_signal, "ann" : resampled_annotation})
            
        self.samples_list = []
        self.label_list = []
        self.qrs_samples = []
        self.idx_AFIB = []
        self.idx_Normal = []
        # self.label = []
        self.number_AFIB = []
        self.number_Normal = []
        no_of_afib = 0
        no_of_normal = 0
        for n,dic in enumerate(self.ecg_list):
            print(dic["name"])
            # xqrs = wfdb.processing.XQRS(sig=dic["rec"], fs=fs)
            # xqrs.detect()
            # qrs_inds = xqrs.qrs_inds
            if(len(dic["ann"].sample)==1):
                idx_next = len(dic["rec"])
            else:
                idx_next = dic["ann"].sample[1]
            label_t = dic["ann"].aux_note[0]
            temp_aux = 0
            for idx in range(dic["ann"].sample[0],len(dic["rec"]),200):
                while(idx>=idx_next):
                    temp_aux += 1
                    if(temp_aux!=len(dic["ann"].sample)-1):
                        idx_next = dic["ann"].sample[temp_aux+1]
                    else:
                        idx_next = len(dic["rec"])
                    label_t = dic["ann"].aux_note[temp_aux]
                if(label_t == '(AFIB'):
                    self.idx_AFIB.append(idx)
                    self.number_AFIB.append(n)
                else:
                    self.idx_Normal.append(idx)
                    self.number_Normal.append(n)
                if (label_t == '(AFIB'):
                    no_of_afib+=1
                else:
                    no_of_normal+=1
        self.sampled_idx_normal = np.random.choice(len(self.number_Normal), size=len(self.number_AFIB), replace=True)
    def resample(self):
        self.sampled_idx_normal = np.random.choice(len(self.number_Normal), size=len(self.number_AFIB), replace=True)
                
    def __len__(self):
        return len(self.sampled_idx_normal) + len(self.idx_AFIB)

    def __getitem__(self, idx):
        if(idx<len(self.sampled_idx_normal)):
            sample = extract_segment_with_padding(self.ecg_list[self.number_Normal[self.sampled_idx_normal[idx]]]["rec"], self.idx_Normal[self.sampled_idx_normal[idx]],self.N)
            sample = torch.Tensor(sample).unsqueeze(0)
            return sample, 0
        else:
            idx = idx - len(self.sampled_idx_normal)
            sample = extract_segment_with_padding(self.ecg_list[self.number_AFIB[idx]]["rec"], self.idx_AFIB[idx],self.N)
            sample = torch.Tensor(sample).unsqueeze(0)
            return sample, 1

In [3]:
ds = SHDB(100,5,fs=100)

001
002
003
004
005
006
007
008
009
010
011
012
013
014
015
016
017
018
019
020
021
022
023
024
025
026
027
028
029
030
031
032
033
034
035
036
037
038
039
040
041
042
043
045
046
047
048
049
050
051
052
054
055
056
062
064
065
070
071
073
077
084
086
102
103
105
106
107
108
109
110
111
112
113
114
115
116
117
118
122
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
001
002
003
004
005
006
007
008
009
010
011
012
013
014
015
016
017
018
019
020
021
022
023
024
025
026
027
028
029
030
031
032
033
034
035
036
037
038
039
040
041
042
043
045
046
047
048
049
050
051
052
054
055
056
062
064
065
070
071
073
077
084
086
102
103
105
106
107
108
109
110
111
112
113
114
115
116
117
118
122
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143


In [4]:
from torch.utils.data import DataLoader, random_split
train_set, val_set = random_split(ds, [0.8, 0.2])
train = DataLoader(train_set, batch_size=32, shuffle=True)
val = DataLoader(val_set, batch_size=32, shuffle=True)

## Model a La resnet

In [5]:
class ResNetBlock(nn.Module):
    def __init__(self,in_channels, out_channels):
        """
        output same as input
        """
        super(ResNetBlock, self).__init__()
        self.conv1 = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
                        nn.BatchNorm1d(out_channels),
                        nn.ReLU(inplace=False))  # Changed inplace to False
        self.conv2 = nn.Sequential(
                        nn.Conv1d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
                        nn.BatchNorm1d(out_channels),
                        nn.ReLU(inplace=False))
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        if(in_channels != out_channels):
            self.residual = nn.Sequential(
                nn.Conv1d(self.in_channels, out_channels, kernel_size=1, stride=1),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self,x):
        out = self.conv1(x)
        out = self.conv2(out)
        if self.in_channels != self.out_channels:
            residual = self.residual(x)
        else:
            residual = x
        return F.relu(out + residual, inplace=False)


class ResNetLike(nn.Module):
    def __init__(self, input = 201, input_ch = 1, num_classes = 2):
        super(ResNetLike, self).__init__()
        self.model = nn.Sequential(
            nn.Conv1d(input_ch, 64, kernel_size=7, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            ResNetBlock(64,64),
            ResNetBlock(64,64),
            ResNetBlock(64,64),
            ResNetBlock(64,128),    # out 1 x 128 x n
            nn.MaxPool1d(2),        # out 1 x 128 x n//2
            ResNetBlock(128,128),
            ResNetBlock(128,128),
            ResNetBlock(128,256),
            nn.MaxPool1d(2),        # out 1 x 256 x n//2
            ResNetBlock(256,256),
            ResNetBlock(256,256),
            ResNetBlock(256,512),
            nn.MaxPool1d(2),        # out 1 x 512 x n//8
            nn.Flatten(),
            nn.Linear(512*(input//8), 256),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
        self.model.to('cuda:0')

    def forward(self, x):

        return self.model(x)
    
    def train_model(self, train_loader, valid_loader, num_epochs = 5, learning_rate=0.001, save_best = False, save_thr = 0.94):
        best_accuracy = 0.0
        total_step = len(train_loader)
        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.RMSprop(self.parameters(), lr=learning_rate, weight_decay = 0.005, momentum = 0.9)  

        for epoch in range(num_epochs):
            # self.train()
            correct = 0
            total = 0
            for i, (images, labels) in enumerate(tqdm(train_loader)):
                # Move tensors to the configured device
                images = images.float().to("cuda")
                labels = labels.type(torch.LongTensor)
                labels = labels.to("cuda")


                optimizer.zero_grad()

                # Forward pass
                outputs = self.forward(images)
                loss = criterion(outputs, labels)
                # Backward and optimize
                loss.backward()
                
                optimizer.step()

                # accuracy
                _, predicted = torch.max(outputs.data, 1)
                correct += (torch.eq(predicted, labels)).sum().item()
                total += labels.size(0)

                del images, labels, outputs

            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}, Accuracy: {:.4f}'
                            .format(epoch+1, num_epochs, i+1, total_step, loss.item(), (float(correct))/total))


            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Validation
            with torch.no_grad():
                correct = 0
                total = 0
                for images, labels in valid_loader:
                    images = images.float().to("cuda")
                    labels = labels.to("cuda")
                    outputs = self.forward(images)
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (torch.eq(predicted, labels)).sum().item()
                    del images, labels, outputs
                if(((100 * correct / total) > best_accuracy) and save_best and ((100 * correct / total) > save_thr)):
                    torch.save(self.state_dict(), "best_resnet50_MINST-DVS2.pt")

                print('Accuracy of the network: {} %'.format( 100 * correct / total))

In [6]:
model_res = ResNetLike()


In [7]:
model_res.train_model(train,val,num_epochs=5)

100%|██████████| 41922/41922 [11:50<00:00, 59.03it/s]


Epoch [1/5], Step [41922/41922], Loss: 0.4352, Accuracy: 0.8588
Accuracy of the network: 86.62269585596975 %


100%|██████████| 41922/41922 [11:45<00:00, 59.39it/s]


Epoch [2/5], Step [41922/41922], Loss: 0.2627, Accuracy: 0.8705
Accuracy of the network: 83.94359729734565 %


100%|██████████| 41922/41922 [11:45<00:00, 59.46it/s]


Epoch [3/5], Step [41922/41922], Loss: 0.1726, Accuracy: 0.8700
Accuracy of the network: 88.84916540936388 %


100%|██████████| 41922/41922 [11:45<00:00, 59.41it/s]


Epoch [4/5], Step [41922/41922], Loss: 0.4813, Accuracy: 0.8704
Accuracy of the network: 88.03007985115126 %


100%|██████████| 41922/41922 [12:23<00:00, 56.36it/s]


Epoch [5/5], Step [41922/41922], Loss: 0.1691, Accuracy: 0.8704
Accuracy of the network: 90.27891249768915 %


```
Epoch [1/90], Step [41922/41922], Loss: 0.2852, Accuracy: 0.9416
Accuracy of the network: 94.53356551193593 %
  2%|▏         | 876/41922 [00:14<11:42, 58.41it/s]

Epoch [1/90], Step [41922/41922], Loss: 0.2852, Accuracy: 0.9416
Accuracy of the network: 94.53356551193593 %
  2%|▏         | 876/41922 [00:14<11:42, 58.41it/s]

```

In [9]:
model_res.train_model(train,val,num_epochs=90,learning_rate=0.0001,save_best=True)

100%|██████████| 41922/41922 [12:20<00:00, 56.61it/s]


Epoch [1/90], Step [41922/41922], Loss: 0.0333, Accuracy: 0.9474
Accuracy of the network: 94.87825532092529 %


  9%|▉         | 3745/41922 [01:02<10:38, 59.76it/s]


KeyboardInterrupt: 